<a href="https://colab.research.google.com/github/warrensuca/chinese-recipe-api/blob/main/clustering/cluster_recipes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import PowerTransformer
import matplotlib.pyplot as plt
import joblib
print('setup complete')

Matplotlib is building the font cache; this may take a moment.


setup complete


In [3]:
df_recipes = pd.read_csv('cleaned_recipes.csv')
nutrition_features = ['Calories', 'Carbohydrates', 'Protein', 'Fat', 'Saturated Fat', 'Sodium', 'Sugar']

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_recipes.csv'

#Clustering

##Elbow Method
Find the optimal K

In [ ]:
inertia = []
k_range = range(1, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=0, n_init='auto')
    kmeans.fit(df_recipes[nutrition_features])
    inertia.append(kmeans.inertia_)

# Plot the Elbow curve
plt.plot(k_range, inertia, marker='o')
plt.title('The Elbow Method')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.show()

##Cluster recipes and interpret centroids

In [ ]:
df_recipes = pd.read_csv('cleaned_raw_recipes.csv')

kmeans = KMeans(n_clusters=5, random_state=0)

pt = PowerTransformer(method='yeo-johnson')


scaled_features = pt.fit_transform(df_recipes[nutrition_features])

#create non-scaled and scaled columns
for index, name in enumerate(nutrition_features):
  df_recipes[f'Scaled_{name}'] = scaled_features[:, index]
df_recipes['Cluster'] = kmeans.fit_predict(scaled_features)


scaled_centers = kmeans.cluster_centers_


real_world_centers = pt.inverse_transform(scaled_centers)


cluster_profiles = pd.DataFrame(real_world_centers, columns=nutrition_features)
cluster_profiles.index = [f"Cluster {i}" for i in range(5)]


display(cluster_profiles.round(1))


##Seem to be distinct archetypes
- sides
- protein/meat heavy mains
- sweets
- balanced meals
- keto friendly

In [ ]:
cluster_names = {
    0: "Light Sides & Soups",
    1: "Rich Meat Mains",
    2: "Desserts & Sweets",
    3: "Balanced Carb-Mains",
    4: "Low-Carb Proteins"
}


df_recipes['Cluster_Name'] = df_recipes['Cluster'].map(cluster_names)

df_recipes.head()


##Assess results
Use cluster cardinality, cluster magnitude, then graph against each other to interpret results

In [ ]:
#x axis is categories, y axis is # of examples under this cluster
names = list(cluster_names.keys())
points_in_cluster = df_recipes['Cluster_Name'].value_counts()


plt.bar(names, points_in_cluster, color='skyblue', edgecolor='black', width=0.6)


plt.title('Cardinality')
plt.xlabel('Cluster Number from 0 to 4')
plt.ylabel('Points in Cluster')


plt.show()


In [ ]:
output_file = "clustered_recipes.csv"

df_recipes.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(f"Output file: {output_file}")

In [ ]:
joblib.dump(pt, 'scaler.joblib')
joblib.dump(kmeans, 'kmeans.joblib')